# Personality Atlas Embedding Projector

Interactive visualization of 6,746 personality trait embeddings across the 44-model atlas.

**Data source:** HuggingFace `Wildertrek/personality-atlas-3072`  
**Embeddings:** OpenAI text-embedding-3-large (3072-dim)  
**Models:** OCEAN, HEXACO, NPI, BDI, GAD-7, MCMI, and 38 others

Run this notebook in Google Colab with no setup required (all dependencies auto-installed).

## Section 1: Setup & Install

In [ ]:
!pip install -q plotly scikit-learn umap-learn datasets huggingface_hub numpy pandas

## Section 2: Load from HuggingFace

In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score
import umap
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

print("📥 Loading personality atlas from HuggingFace...")
ds = load_dataset("Wildertrek/personality-atlas-3072")
df_raw = ds['train'].to_pandas()

print(f"✅ Loaded {len(df_raw)} trait embeddings")
print(f"   Dimensions: {len(df_raw['embedding'][0])}-dim")
print(f"   Models: {df_raw['model'].nunique()}")
print(f"   Categories: {df_raw['category'].nunique()}")

# Display sample
print("\n📋 Sample rows:")
display(df_raw[['model', 'category', 'factor', 'adjective', 'synonym']].head(10))

## Section 3: Extract Embeddings & Prepare Data

In [ ]:
# Convert embedding lists to numpy array
X = np.array(df_raw['embedding'].tolist(), dtype=np.float32)

# Create metadata dataframe (drop embedding column for clarity)
df = df_raw[['model', 'category', 'factor', 'adjective', 'synonym', 'verb', 'noun']].copy()
df['idx'] = range(len(df))

print(f"✅ Embeddings shape: {X.shape}")
print(f"   Memory: {X.nbytes / 1e9:.2f} GB")

# Category metadata with distinct colors
CATEGORY_COLORS = {
    "Trait-Based": "#1f77b4",
    "Narcissism-Based": "#9467bd",
    "Motivational": "#2ca02c",
    "Cognitive": "#ff7f0e",
    "Clinical": "#d62728",
    "Interpersonal": "#e377c2",
    "App/Holistic": "#7f7f7f",
}

# Model colors - 44 distinct colors with high contrast
MODEL_COLORS = {
    "16PF": "#1f77b4", "OCEAN": "#2ca02c", "HEXACO": "#d62728", "EPM": "#ff7f0e", "MBTI": "#9467bd", "FTM": "#8c564b",
    "NPI": "#e377c2", "NARQ": "#7f7f7f", "FFNI": "#bcbd22", "HSNS": "#17becf", "PNI": "#1f77b4",
    "MCMI-N": "#ff7f0e", "FFNI-SF": "#2ca02c", "IPN": "#d62728", "DT3": "#9467bd", "DT4": "#8c564b",
    "AAM": "#e377c2", "MST": "#7f7f7f", "RFT": "#bcbd22", "SDT": "#17becf", "STBV": "#1f77b4", "CS": "#ff7f0e",
    "PCT": "#2ca02c", "SCM": "#d62728", "CEST": "#9467bd", "FSLS": "#8c564b",
    "MMPI": "#e377c2", "TCI": "#7f7f7f", "TMP": "#bcbd22", "BDI": "#17becf", "GAD-7": "#1f77b4",
    "SCID": "#ff7f0e", "MCMI": "#2ca02c", "Rorschach": "#d62728", "TAT": "#9467bd", "WAIS": "#8c564b",
    "TKI": "#e377c2", "DISC": "#7f7f7f",
    "RIASEC": "#bcbd22", "BT": "#17becf", "TEI": "#1f77b4", "EM": "#ff7f0e", "PAPC": "#2ca02c", "CMOA": "#d62728",
}

print(f"\n📊 Category distribution:")
print(df['category'].value_counts())

## Section 4: PCA Projection (2D + 3D)

In [ ]:
print("🔄 Computing PCA...")
pca = PCA(n_components=50)
X_pca = pca.fit_transform(X).astype(np.float32)

print(f"✅ PCA variance explained: {100*pca.explained_variance_ratio_.sum():.1f}%")
print(f"   PC1: {100*pca.explained_variance_ratio_[0]:.1f}%")
print(f"   PC2: {100*pca.explained_variance_ratio_[1]:.1f}%")
print(f"   PC3: {100*pca.explained_variance_ratio_[2]:.1f}%")

# 3D PCA with larger figure and better styling
fig = go.Figure()
for cat in sorted(df['category'].unique()):
    mask = df['category'] == cat
    fig.add_trace(go.Scatter3d(
        x=X_pca[mask, 0],
        y=X_pca[mask, 1],
        z=X_pca[mask, 2],
        mode='markers',
        name=cat,
        marker=dict(
            size=3,
            color=CATEGORY_COLORS.get(cat, '#999999'),
            opacity=0.7,
            line=dict(width=0),
        ),
        text=[f"<b>{row['adjective']}</b><br>Factor: {row['factor']}<br>Model: {row['model']}" 
              for _, row in df[mask].iterrows()],
        hovertemplate='%{text}<extra></extra>',
    ))

fig.update_layout(
    title='Personality Atlas: PCA 3D Projection (50 components)',
    scene=dict(
        xaxis_title=f"PC1 ({100*pca.explained_variance_ratio_[0]:.1f}%)",
        yaxis_title=f"PC2 ({100*pca.explained_variance_ratio_[1]:.1f}%)",
        zaxis_title=f"PC3 ({100*pca.explained_variance_ratio_[2]:.1f}%)",
    ),
    height=900,
    template='plotly_dark',
    font=dict(size=12),
    legend=dict(x=0.98, y=0.98, bgcolor="rgba(20,20,20,0.8)", bordercolor="white", borderwidth=1),
)
fig.show()

print("\n💾 Saving PCA 3D figure...")
fig.write_html('pca_3d.html')
print("✅ Saved to pca_3d.html")

## Section 5: t-SNE Projection

In [ ]:
print("🔄 Computing t-SNE (pre-reducing to PCA-50 for speed)...")
pca_50 = PCA(n_components=50)
X_pca_50 = pca_50.fit_transform(X).astype(np.float32)

tsne = TSNE(n_components=2, perplexity=25, learning_rate=10, random_state=42, max_iter=1000, verbose=1)
X_tsne = tsne.fit_transform(X_pca_50).astype(np.float32)
print(f"✅ t-SNE computed (2D)")

# Compute category centroids
centroids = {}
for cat in df['category'].unique():
    mask = df['category'] == cat
    centroids[cat] = X_tsne[mask].mean(axis=0)

# t-SNE 2D visualization with improved styling
fig = go.Figure()

for cat in sorted(df['category'].unique()):
    mask = df['category'] == cat
    fig.add_trace(go.Scatter(
        x=X_tsne[mask, 0],
        y=X_tsne[mask, 1],
        mode='markers',
        name=cat,
        marker=dict(
            size=4,
            color=CATEGORY_COLORS.get(cat, '#999999'),
            opacity=0.7,
            line=dict(width=0),
        ),
        text=[f"<b>{row['adjective']}</b><br>Factor: {row['factor']}<br>Model: {row['model']}" 
              for _, row in df[mask].iterrows()],
        hovertemplate='%{text}<extra></extra>',
    ))

# Add category centroids as stars
centroid_names = list(centroids.keys())
centroid_coords = np.array(list(centroids.values()))
fig.add_trace(go.Scatter(
    x=centroid_coords[:, 0],
    y=centroid_coords[:, 1],
    mode='markers+text',
    name='Category centroids',
    marker=dict(size=12, color='white', symbol='star', line=dict(width=2, color='black')),
    text=centroid_names,
    textposition='top center',
    hovertemplate='%{text}<extra></extra>',
    showlegend=True,
))

fig.update_layout(
    title='Personality Atlas: t-SNE 2D Projection',
    xaxis_title='t-SNE 1',
    yaxis_title='t-SNE 2',
    height=900,
    template='plotly_dark',
    font=dict(size=12),
    legend=dict(x=0.98, y=0.98, bgcolor="rgba(20,20,20,0.8)", bordercolor="white", borderwidth=1),
)
fig.show()

print("\n💾 Saving t-SNE figure...")
fig.write_html('tsne_2d.html')
print("✅ Saved to tsne_2d.html")

## Section 6: UMAP Projection (2D)

In [ ]:
print("🔄 Computing UMAP (pre-reducing to PCA-50 for speed)...")
reducer = umap.UMAP(n_components=2, n_neighbors=15, random_state=42, verbose=1)
X_umap = reducer.fit_transform(X_pca_50).astype(np.float32)
print(f"✅ UMAP computed (2D, neighbors=15)")

# UMAP 2D visualization with improved styling matching PCA/t-SNE
fig = go.Figure()
for cat in sorted(df['category'].unique()):
    mask = df['category'] == cat
    fig.add_trace(go.Scatter(
        x=X_umap[mask, 0],
        y=X_umap[mask, 1],
        mode='markers',
        name=cat,
        marker=dict(
            size=4,
            color=CATEGORY_COLORS.get(cat, '#999999'),
            opacity=0.7,
            line=dict(width=0),
        ),
        text=[f"<b>{row['adjective']}</b><br>Factor: {row['factor']}<br>Model: {row['model']}" 
              for _, row in df[mask].iterrows()],
        hovertemplate='%{text}<extra></extra>',
    ))

fig.update_layout(
    title='Personality Atlas: UMAP 2D Projection (PCA-50 pre-reduction, neighbors=15)',
    xaxis_title='UMAP 1',
    yaxis_title='UMAP 2',
    height=900,
    template='plotly_dark',
    font=dict(size=12),
    legend=dict(x=0.98, y=0.98, bgcolor="rgba(20,20,20,0.8)", bordercolor="white", borderwidth=1),
)
fig.show()

print("\n💾 Saving UMAP figure...")
fig.write_html('umap_2d.html')
print("✅ Saved to umap_2d.html")

## Section 7: Per-Category Cluster Analysis

In [ ]:
from plotly.subplots import make_subplots

categories = sorted(df['category'].unique())
n_cats = len(categories)

# Create subplots (one per category)
specs = [[{"type": "scatter"}] * 2 for _ in range((n_cats + 1) // 2)]
fig = make_subplots(
    rows=(n_cats + 1) // 2,
    cols=2,
    subplot_titles=categories,
)

for idx, cat in enumerate(categories):
    row = (idx // 2) + 1
    col = (idx % 2) + 1
    
    mask = df['category'] == cat
    
    # Color by model using MODEL_COLORS for consistency
    models = sorted(df[mask]['model'].unique())
    
    for model in models:
        model_mask = mask & (df['model'] == model)
        model_color = MODEL_COLORS.get(model, '#999999')
        
        fig.add_trace(
            go.Scatter(
                x=X_tsne[model_mask, 0],
                y=X_tsne[model_mask, 1],
                mode='markers',
                name=model,
                marker=dict(size=4, color=model_color, opacity=0.7, line=dict(width=0)),
                showlegend=(idx == 0),  # Only show legend for first subplot
                hovertemplate='<b>%{text}</b><extra></extra>',
                text=[f"{r['adjective']}" for _, r in df[model_mask].iterrows()],
            ),
            row=row,
            col=col,
        )

fig.update_xaxes(title_text='t-SNE 1', row=(n_cats + 1) // 2, col=1)
fig.update_yaxes(title_text='t-SNE 2', row=(n_cats + 1) // 2, col=1)

fig.update_layout(
    title_text="Per-Category Clustering (t-SNE coordinates, colored by model)",
    height=900 if n_cats <= 4 else 1200,
    width=1400,
    template='plotly_dark',
    font=dict(size=11),
    showlegend=True,
    legend=dict(x=1.02, y=1.0, bgcolor="rgba(20,20,20,0.8)", bordercolor="white", borderwidth=1),
)
fig.show()

print("\n💾 Saving per-category analysis...")
fig.write_html('per_category_tsne.html')
print("✅ Saved to per_category_tsne.html")

## Section 8: Category Centroid Similarity Heatmap

In [ ]:
print("🔄 Computing category centroid similarities...")

# Compute centroids in embedding space
centroids_embed = {}
for cat in categories:
    mask = df['category'] == cat
    cat_embed = X[mask]
    centroid = cat_embed.mean(axis=0)
    centroid = centroid / (np.linalg.norm(centroid) + 1e-10)  # Normalize
    centroids_embed[cat] = centroid

# Compute pairwise cosine similarity
centroids_array = np.array(list(centroids_embed.values()))
sim_matrix = centroids_array @ centroids_array.T

# Create heatmap with improved styling
fig = px.imshow(
    sim_matrix,
    x=categories,
    y=categories,
    color_continuous_scale='RdBu',
    zmin=-1,
    zmax=1,
    labels=dict(color='Cosine<br>Similarity'),
    title='Category Centroid Similarity Matrix (3072-Dim Embedding Space)',
    aspect='auto',
)

# Add text annotations for values
fig.update_traces(
    text=np.round(sim_matrix, 3),
    texttemplate='%{text}',
    textfont=dict(size=10),
)

fig.update_layout(
    height=700,
    width=750,
    template='plotly_dark',
    font=dict(size=11),
    xaxis_title='Category',
    yaxis_title='Category',
)
fig.show()

print("\n💾 Saving heatmap...")
fig.write_html('category_similarity_heatmap.html')
print("✅ Saved to category_similarity_heatmap.html")

# Print similarity statistics
print("\n📊 Centroid Similarities (most similar pairs):")
similarities_list = []
for i, cat1 in enumerate(categories):
    for j, cat2 in enumerate(categories):
        if i < j:  # Only upper triangle
            sim = sim_matrix[i, j]
            similarities_list.append((sim, cat1, cat2))

# Sort by similarity (highest first)
similarities_list.sort(reverse=True)
for sim, cat1, cat2 in similarities_list[:10]:
    print(f"  {cat1:20s} ↔ {cat2:20s}: {sim:+.3f}")

## Section 9: Separation Metrics

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

print("🔄 Computing cluster separation and quality metrics...")

# Map categories to labels
labels = df['category'].values
unique_cats = sorted(np.unique(labels))
label_map = {cat: i for i, cat in enumerate(unique_cats)}
label_ints = np.array([label_map[cat] for cat in labels])

# Global metrics
silhouette = silhouette_score(X_pca_50, label_ints)
db_score = davies_bouldin_score(X_pca_50, label_ints)
ch_score = calinski_harabasz_score(X_pca_50, label_ints)

print("\n" + "="*60)
print("GLOBAL CLUSTER QUALITY METRICS")
print("="*60)
print(f"\n✅ Silhouette Score: {silhouette:.4f}")
print("   Interpretation: -1 to 1; higher is better (well-separated clusters)")
print(f"   Current: {'EXCELLENT' if silhouette > 0.5 else 'GOOD' if silhouette > 0.3 else 'MODERATE'}")

print(f"\n✅ Davies-Bouldin Index: {db_score:.4f}")
print("   Interpretation: Lower is better (0 = perfect separation)")
print(f"   Current: {'EXCELLENT' if db_score < 1.0 else 'GOOD' if db_score < 2.0 else 'MODERATE'}")

print(f"\n✅ Calinski-Harabasz Score: {ch_score:.2f}")
print("   Interpretation: Higher is better (good cluster definition)")

# Per-category silhouette scores
print("\n" + "="*60)
print("PER-CATEGORY SILHOUETTE SCORES")
print("="*60)
cat_scores = {}
for cat in categories:
    mask = label_ints == label_map[cat]
    n_samples = mask.sum()
    if n_samples > 1:
        score = silhouette_score(X_pca_50[mask], label_ints[mask])
        cat_scores[cat] = score
        print(f"  {cat:20s} (n={n_samples:4d}): {score:+.4f}")

# Summary statistics
print("\n" + "="*60)
print("ATLAS COMPOSITION")
print("="*60)
print(f"  Total embeddings: {len(df):,}")
print(f"  Embedding dimension: 3072 (OpenAI text-embedding-3-large)")
print(f"  Number of models: {df['model'].nunique()}")
print(f"  Number of categories: {len(categories)}")
print(f"  Mean embeddings per model: {len(df) / df['model'].nunique():.1f}")
print(f"  Mean embeddings per category: {len(df) / len(categories):.1f}")

# Distribution by category
print("\n" + "="*60)
print("EMBEDDINGS BY CATEGORY")
print("="*60)
category_counts = df['category'].value_counts().sort_values(ascending=False)
for cat, count in category_counts.items():
    pct = 100 * count / len(df)
    print(f"  {cat:20s}: {count:4d} ({pct:5.1f}%)")

## Summary

✅ **Generated Figures:**
- `pca_3d.html` — PCA 3D visualization
- `tsne_2d.html` — t-SNE 2D global clustering
- `umap_2d.html` — UMAP 2D projection
- `per_category_tsne.html` — Per-category cluster analysis
- `category_similarity_heatmap.html` — Centroid similarity matrix

**Key Insights:**
- **Silhouette Score** measures how well-separated categories are
- **Davies-Bouldin Index** quantifies cluster compactness and separation
- **Category Centroids** show semantic relationships across 7 personality dimensions
- **Per-Model Analysis** reveals which models cluster together vs. diverge

Use these visualizations in presentations, papers, or interactive dashboards.